# BIRCH Clustering Analysis

Balanced Iterative Reducing and Clustering using Hierarchies (BIRCH) for student behavior segmentation.

## Overview
- Load encoded dataset from `encoded_unsupervised.csv`
- Apply BIRCH clustering
- Evaluate clusters using silhouette score and other metrics
- Visualize and interpret clusters

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import Birch
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")


## 1. Load Data

Load the preprocessed and scaled dataset.


In [ ]:
# Load encoded dataset
DATA_PATH = '../data/final/encoded_unsupervised.csv'
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()


## 2. BIRCH Clustering

BIRCH parameters:
- **threshold**: Maximum radius of a subcluster (controls cluster granularity)
- **branching_factor**: Maximum number of subclusters per node
- **n_clusters**: Final number of clusters (None = use subclusters directly)


In [ ]:
# Initialize and fit BIRCH
birch = Birch(
    threshold=0.5,
    branching_factor=50,
    n_clusters=None  # Let BIRCH determine optimal subclusters first
)

# Fit and predict
cluster_labels = birch.fit_predict(df)

# Add cluster labels to dataframe
df_clustered = df.copy()
df_clustered['cluster'] = cluster_labels

print(f"Number of clusters found: {len(np.unique(cluster_labels))}")
print(f"\nCluster distribution:")
print(df_clustered['cluster'].value_counts().sort_index())


## 3. Cluster Evaluation

Evaluate clustering quality using:
- **Silhouette Score**: Measures how similar points are to their own cluster vs other clusters (-1 to 1, higher is better)
- **Calinski-Harabasz Index**: Ratio of between-cluster to within-cluster variance (higher is better)
- **Davies-Bouldin Index**: Average similarity between clusters (lower is better)


In [ ]:
# Calculate evaluation metrics
silhouette = silhouette_score(df, cluster_labels)
calinski = calinski_harabasz_score(df, cluster_labels)
davies = davies_bouldin_score(df, cluster_labels)

print("=" * 50)
print("CLUSTERING EVALUATION METRICS")
print("=" * 50)
print(f"\nSilhouette Score:        {silhouette:.4f}")
print(f"Calinski-Harabasz Index: {calinski:.4f}")
print(f"Davies-Bouldin Index:    {davies:.4f}")


## 4. Cluster Visualization

Use PCA to reduce dimensions for 2D visualization.


In [ ]:
# Reduce to 2D using PCA
pca = PCA(n_components=2)
df_pca = pca.fit_transform(df)

print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_):.2%}")

# Plot clusters
plt.figure(figsize=(10, 8))
scatter = plt.scatter(df_pca[:, 0], df_pca[:, 1], 
                      c=cluster_labels, cmap='viridis', 
                      alpha=0.5, s=10)
plt.colorbar(scatter, label='Cluster')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('BIRCH Clustering Results (PCA Projection)')
plt.tight_layout()
plt.show()


## 5. Cluster Profiling

Analyze the characteristics of each cluster by examining feature means.


In [ ]:
# Calculate cluster statistics
cluster_stats = df_clustered.groupby('cluster').mean()

print("=" * 50)
print("CLUSTER PROFILES (Feature Means)")
print("=" * 50)
print(f"\nNumber of samples per cluster:")
print(df_clustered['cluster'].value_counts().sort_index())
print(f"\n")
cluster_stats.T


In [ ]:
# Heatmap of cluster profiles
plt.figure(figsize=(14, 10))
sns.heatmap(cluster_stats.T, annot=True, fmt='.2f', cmap='RdYlBu_r', 
            center=0, linewidths=0.5)
plt.title('Cluster Profiles - Feature Means')
plt.xlabel('Cluster')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()
